In [2]:
import pandas as pd
import numpy as np
import joblib
import json

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score



In [3]:
df = pd.read_csv("data/processed/featured_train.csv")
df = df.sort_values(by=["Year", "Month", "Day"]).reset_index(drop=True)

X = df.drop("Sales", axis=1)
y = df["Sales"]
y_log = np.log1p(y)

split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y_log.iloc[:split_index]
y_test = y_log.iloc[split_index:]

print("Prepared evaluation data")

Prepared evaluation data


In [4]:
model = joblib.load("artifacts/model.pkl")


In [5]:
y_pred_log = model.predict(X_test)

y_test_actual = np.expm1(y_test)
y_pred_actual = np.expm1(y_pred_log)

In [7]:
mae = mean_absolute_error(y_test_actual, y_pred_actual)
rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))
r2 = r2_score(y_test_actual, y_pred_actual)

mape = np.mean(
    np.abs((y_test_actual - y_pred_actual) / np.clip(y_test_actual, 1, None))
) * 100

print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)
print("MAPE:", mape)

MAE: 770.7407597111405
RMSE: 1136.99376268698
R2 Score: 0.862220894002356
MAPE: 10.724341136971313


In [9]:
metrics = {
    "MAE": float(mae),
    "RMSE": float(rmse),
    "R2": float(r2),
    "MAPE": float(mape)
}

with open("artifacts/metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

print("Metrics saved to /artifacts/metrics.json")
metrics

Metrics saved to /artifacts/metrics.json


{'MAE': 770.7407597111405,
 'RMSE': 1136.99376268698,
 'R2': 0.862220894002356,
 'MAPE': 10.724341136971313}

In [10]:
importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
}).sort_values(by="Importance", ascending=False)

importance_df.head(20)

,Feature,Importance
20,StoreType_b,0.209097
2,Promo,0.186101
24,Assortment_c,0.071183
4,CompetitionDistance,0.065553
5,Promo2,0.059999
21,StoreType_c,0.059366
22,StoreType_d,0.052723
0,Store,0.049937
15,Promo2RunningYears,0.034429
14,CompetitionOpenYears,0.031963


In [11]:
comparison_df = pd.DataFrame({
    "Actual Sales": y_test_actual.values[:20],
    "Predicted Sales": y_pred_actual[:20]
})

comparison_df

,Actual Sales,Predicted Sales
0,8120.0,8046.535156
1,11129.0,7999.138184
2,6175.0,7317.062500
3,7505.0,8218.714844
4,7192.0,6544.177246
5,8952.0,7730.839844
6,9465.0,8124.176758
7,7892.0,7392.818359
8,7725.0,7492.841797
9,7895.0,7370.000488
